# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, *range(500, 600)}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None

    for attempt in range(tries):
      try:
        r = requests.get(BASE, params=params, timeout=15)
      except requests.RequestException as e:
          last = e
      else:
          if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])
          if r.status_code not in RETRY_STATUSES:
            raise RuntimeError(
                f'{BASE} returned {r.status_code}: {r.text[:200]}'
            )

          last = f'{r.status_code}: {r.text[:200]}'

      if attempt < tries - 1:
              delay = 0.5 * (2 ** attempt)
              print(f'attempt {attempt +1} failed ({last}); retrying in {delay}s')

              time.sleep(delay)

    raise RuntimeError(f'all {tries} attempts failed: {last}')

    # TODO: loop up to `tries` times
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed


Set up the 'get_weather' table to find the max temperature and total. precipitation for each day requested. Created a 'for' loop to pull data from meteo API, with fall backs if requests fail.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about

used_fallback = False

weather = get_weather('2026-09-05', '2026-10-03')

weather = weather[
    weather['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

weather



,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
28,2026-10-03,22.0,14.7


In [3]:
from datetime import datetime

pulled_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(pulled_at)

from datetime import datetime

pulled_at = datetime.now().astimezone().isoformat(timespec='seconds')

2026-10-12T01:58:04+00:00


I used one request to fetch the weather within the daye range and then used .isin to esnure teh time and date corressponded. I used .copy() to make a new table so cleaning is easier.


I added 'pulled_at' so I am able to see when I pulled the data from the weather API.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [4]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
  wx = FALLBACK.copy()
else:
  wx = weather.copy()


wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'

wx

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


Because used_fallback = false from Part 2 the statement jumps to 'else' and uses the API data from 'wx'. This is why the 'source' in the table displays as 'open-meteo'.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [5]:
# TODO

wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


Used '.rename' to change the column names from the raw API so it will be cleaner to use in a join function. Used .to_datatime' convert the text into data values, and then reordered the coloumns for the table.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [6]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


Uncommneted all of the asserts to make sure the weather table was ready to join.

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
# TODO

wettest = wx.loc[wx['temp_max'].idxmax()]
driest = wx.loc[wx['precip_mm'].idxmax()]

difference = wettest['precip_mm'] - driest['precip_mm']

if difference == 0:
  print('The dates had the same total precipitation.')
else:
  print(
      f"{wettest['date']:%Y-%m-%d} was wetter than "
      f"{driest['date']:%Y-%m-%d} by {difference:.1f} mm."

  )
if used_fallback:
  print('Comparison used fallback data')

2026-09-05 was wetter than 2026-10-03 by -12.2 mm.


Calculated the wettest and driest day of range to then find the difference between the two. I created an 'if' statement in case the participation is the same, and had an 'else' statement that was used to print the comparison.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
# TODO

print('Endpoint:', BASE)
print('Coordinates: latitude 38.03, longitude -78.51')
print('Date range:', '2026-09-05', 'to', '2026-10-03')
print('Retained Dates: 2026-09-05, 2026-10-03')

if used_fallback:
  print('Pulled at: not applicable - provided sample data')
  print('Source: fallback sample')

else:
  print('Pulled at:', pulled_at)
  print('Source: open-meteo')


Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude 38.03, longitude -78.51
Date range: 2026-09-05 to 2026-10-03
Retained Dates: 2026-09-05, 2026-10-03
Pulled at: 2026-10-12T01:58:04+00:00
Source: open-meteo


Created print commands and then make an if-else statement to verify that the data being printed was from the 'open-meteo' source that has been confirmed by other Parts of the lab.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

A chart code have one of two misakes based on if the data is then used in a left merge or an inner merge. With a left merge the 200 days of sales data would drop the missing weather rows. This could lead to the rainfall appearing as it was zero everyday and it was completely dry. With an inner merge, these empty weather rows would be removed. Both of these errors would be difficult to notice because there would not be an error. This could causer someone to completey misinturpt the relationship between rainfall and sales, as they would be mislead to think there was a relationship between the lack of rain and displayed sales.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
# stretch

orlando = get_weather(
    '2026-09-05', '2026-10-03',
    lat = 28.54, lon = -81.38
)

orlando = orlando[
    orlando['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

orlando['date'] = pd.to_datetime(orlando['date'])

weather_cities = pd.concat([
    wx.assign(city= 'Charlottesville'),
    orlando.assign(city= 'Orlando', source ='open-meteo'),
    ], ignore_index = True)

assert len(weather_cities) ==4
assert not weather_cities.duplicated(['city', 'date']).any()
assert weather_cities['precip_mm'].notna().all()
assert (weather_cities['precip_mm'] >= 0).all()

weather_cities

,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
1,2026-10-03,22.0,14.7,open-meteo,Charlottesville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-03,30.4,14.5,open-meteo,Orlando


The first thing I did was pull Orlando's data by using the get_weather table and changing the coordinates. I then made sure the two tables were for the same range of dates and used the the same column names. I then labeled the data to coresspond to each city and ran assert statements to make sure there werent errors I did not notice.